# CanAI Café — Data Cleaning
**Owner:** Member A  
**Output:** `../data/clean/data_clean.csv`  
**Audit log:** `../data/clean/cleaning_audit_log.csv`

> Run cells top-to-bottom in order. Each step appends to `cleaning_log` so the full record of changes is available at the end.

## 0. Imports

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load Raw Data

In [30]:
RAW_PATH = '../data/raw/Team A CanAI Cafe dataset.xlsx'

df = pd.read_excel(RAW_PATH)

# Snapshot of raw shape — used in the final comparison
_original_rows = len(df)
_original_cols = df.shape[1]

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

Loaded: 10,000 rows × 9 columns


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province
0,TXN_9687814,coffee,2,3.5,7.0,Digital Wallet,In-store,2023-09-28 00:00:00,British Columbia
1,TXN_7002925,Refresher,2,5.0,10.0,Cash,NaN,2023-05-02 00:00:00,NL
2,TXN_7668262,Donut,1,2.0,2.0,Digital Wallet,In-store,2023-11-27 00:00:00,British Columbia
3,TXN_8322624,Refresher,3,5.0,15.0,ERR_PM_102,In-store,NaN,Saskatchewan
4,TXN_9390285,Coffee,1,3.5,3.5,ERR_PM_102,Takeaway,2023-05-11 00:00:00,newfoundland and labrador


## 2. Audit Log Setup

In [31]:
# Every cleaning action appends one entry here.
# Exported to data/clean/cleaning_audit_log.csv at the end of the notebook.
cleaning_log = []

def log_change(step, column, issue, action, rows_affected, notes=''):
    """Record one cleaning action and print a summary line."""
    cleaning_log.append({
        'Step':          step,
        'Column':        column,
        'Issue':         issue,
        'Action':        action,
        'Rows Affected': rows_affected,
        'Notes':         notes,
    })
    suffix = f'  ({notes})' if notes else ''
    print(f"  [{step}] {column}: {action}  →  {rows_affected:,} affected{suffix}")

## 3. Initial Assessment

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    10000 non-null  str    
 1   Item              9913 non-null   str    
 2   Quantity          9922 non-null   object 
 3   Price Per Unit    10000 non-null  float64
 4   Total Spent       10000 non-null  float64
 5   Payment Method    9568 non-null   str    
 6   Location          9285 non-null   str    
 7   Transaction Date  9824 non-null   object 
 8   Province          9650 non-null   str    
dtypes: float64(2), object(2), str(5)
memory usage: 703.3+ KB


In [33]:
null_summary = pd.DataFrame({
    'Null Count': df.isnull().sum(),
    '% of Rows':  (df.isnull().sum() / len(df) * 100).round(2),
})
display(null_summary[null_summary['Null Count'] > 0])

,Null Count,% of Rows
Item,87,0.87
Quantity,78,0.78
Payment Method,432,4.32
Location,715,7.15
Transaction Date,176,1.76
Province,350,3.50


In [34]:
print("=== Numeric columns ===")
display(df[['Price Per Unit', 'Total Spent']].describe())

print("\n=== Categorical value counts ===")
for col in ['Item', 'Payment Method', 'Location', 'Province']:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string())

=== Numeric columns ===


,Price Per Unit,Total Spent
count,10000.000000,10000.000000
mean,4.344850,8.628800
std,2.201565,6.873291
min,2.000000,2.000000
25%,3.000000,3.500000
50%,3.500000,7.000000
75%,5.000000,10.500000
max,9.000000,45.000000



=== Categorical value counts ===

--- Item ---
Item
Coffee       2248
Tea          1257
Sandwich     1060
Cookie        960
Donut         936
Refresher     810
Salad         748
Juice         523
Sandwhich     119
cofee         112
coffe         112
sandwich      111
coffee        102
Cofee         100
C0ffee         97
NaN            87
Tee            78
tee            78
Tea            77
donut          67
Doughnut       64
TEA            63
Donutt         56
Juic           44
juice          43
Juicee         35
               13

--- Payment Method ---
Payment Method
Credit Card       3885
Digital Wallet    3001
Cash              2559
NaN                432
                    95
ERR_PM_102          28

--- Location ---
Location
In-store    5915
Takeaway    3207
NaN          715
             163

--- Province ---
Province
British Columbia             2780
Newfoundland                 1992
Saskatchewan                 1644
Manitoba                     1612
NaN                       

## 4. Handle Missing Values

**Decision framework — drop vs. fill:**

Each column was assessed on whether a null value makes the row analytically usable:

| Column | Null count | Decision | Rationale |
|--------|-----------|----------|-----------|
| Item | 87 | **Drop row** | Cannot identify what was purchased — the row has no analytical value for sales, revenue, or product-mix analysis |
| Quantity | 78 | **Drop row** | Revenue (`Total Spent`) cannot be verified or recalculated without a quantity; the row is unreliable |
| Transaction Date | 176 | **Drop row** | Date is required for any time-series or forecasting work — a dateless transaction cannot be placed on a timeline |
| Payment Method | 432 | **Fill `'Unknown'`** | Payment channel is contextual; the transaction itself (what was bought, when, where) is still valid and worth retaining |
| Location | 715 | **Fill `'Unknown'`** | In-store vs. takeaway is useful for segmentation but not required to analyse revenue or product trends |
| Province | 350 | **Fill `'Unknown'`** | Province will be standardised in Step 7; missing values are retained and labelled for transparency |

**Real-world context:** Dropping rows with missing core fields (Item, Quantity, Date) mirrors standard practice in transactional data pipelines — a sale record missing its product, amount, or timestamp is effectively an incomplete ledger entry and would be flagged for exclusion in any audited financial system. Filling non-critical fields with `'Unknown'` preserves the remaining signal in the row while being transparent about the gap.

In [35]:
print("=== Step 4: Handle Missing Values ===\n")

# 4a — Convert empty/whitespace-only strings to NaN across every column.
# df == '' misses cells that contain only spaces; regex covers both.
_before_nulls = df.isnull().sum().sum()
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
_empty_cells = int(df.isnull().sum().sum() - _before_nulls)
log_change('4a', 'All columns', 'Empty / whitespace-only string cells',
           'Replaced with NaN', _empty_cells,
           'Regex replacement catches both empty strings and space-only cells')

# 4b — Drop rows where Item is null (can't identify what was purchased)
_before = len(df)
df.dropna(subset=['Item'], inplace=True)
log_change('4b', 'Item', 'Null item name', 'Row dropped', _before - len(df),
           'Cannot identify product — row has no analytical value')

# 4c — Drop rows where Quantity is null (can't compute revenue)
_before = len(df)
df.dropna(subset=['Quantity'], inplace=True)
log_change('4c', 'Quantity', 'Null quantity', 'Row dropped', _before - len(df),
           'Revenue calculation requires quantity')

# 4d — Drop rows where Transaction Date is null (required for time-series)
_before = len(df)
df.dropna(subset=['Transaction Date'], inplace=True)
log_change('4d', 'Transaction Date', 'Null date', 'Row dropped', _before - len(df),
           'Date is required for forecasting — row excluded')

# 4e — Payment Method: fill null → 'Unknown' (keep row)
_before = int(df['Payment Method'].isnull().sum())
df['Payment Method'] = df['Payment Method'].fillna('Unknown')
log_change('4e', 'Payment Method', 'Null payment method',
           "Filled with 'Unknown'", _before,
           'Payment channel not always recorded; row retained')

# 4f — Location: fill null → 'Unknown' (keep row)
_before = int(df['Location'].isnull().sum())
df['Location'] = df['Location'].fillna('Unknown')
log_change('4f', 'Location', 'Null location',
           "Filled with 'Unknown'", _before,
           'Location not always captured; row retained')

# 4g — Province: fill null → 'Unknown' (keep row; standardised in step 6)
_before = int(df['Province'].isnull().sum())
df['Province'] = df['Province'].fillna('Unknown')
log_change('4g', 'Province', 'Null province',
           "Filled with 'Unknown'", _before,
           'Will be standardised in step 7')

print(f"\nRows remaining after missing-value handling: {len(df):,}")

=== Step 4: Handle Missing Values ===

  [4a] All columns: Replaced with NaN  →  397 affected  (Regex replacement catches both empty strings and space-only cells)
  [4b] Item: Row dropped  →  100 affected  (Cannot identify product — row has no analytical value)
  [4c] Quantity: Row dropped  →  98 affected  (Revenue calculation requires quantity)
  [4d] Transaction Date: Row dropped  →  211 affected  (Date is required for forecasting — row excluded)
  [4e] Payment Method: Filled with 'Unknown'  →  507 affected  (Payment channel not always recorded; row retained)
  [4f] Location: Filled with 'Unknown'  →  844 affected  (Location not always captured; row retained)
  [4g] Province: Filled with 'Unknown'  →  396 affected  (Will be standardised in step 7)

Rows remaining after missing-value handling: 9,591


## 5. Remove Duplicates & Drop Transaction ID Collisions

**Rationale for dropping duplicate Transaction IDs:** `Transaction ID` is the dataset's primary key and should uniquely identify each transaction. Rows that share a `Transaction ID` but have different items or dates cannot be validated — in a real-world context this pattern can indicate:

- **Fraudulent activity** — a transaction ID reused intentionally to obscure duplicate charges or manipulate revenue records
- **Corrupted data** — a broken ID-generation sequence in the source system producing repeated keys for genuinely separate transactions

Because there is no way to determine which case applies from the data alone, retaining these rows would risk including either fraudulent or corrupted records in downstream analysis. At only 84 rows (~0.88% of the dataset), the data-loss impact is negligible and the integrity gain outweighs retention.

In [36]:
print("=== Step 5: Duplicates ===\n")

# 5a — Drop fully duplicate rows
_before = len(df)
df.drop_duplicates(inplace=True)
log_change('5a', 'All columns', 'Fully duplicate rows', 'Dropped', _before - len(df))

# 5b — Drop rows with duplicate Transaction IDs.
# Transaction ID is a primary key; collisions indicate a source-system error and
# cannot be distinguished from erroneous duplicates, so affected rows are excluded.
_dup_mask = df['Transaction ID'].duplicated(keep=False)
_dup_id_rows = int(_dup_mask.sum())
_before = len(df)
df = df[~_dup_mask].copy()
log_change('5b', 'Transaction ID',
           f'{_dup_id_rows} rows share a non-unique Transaction ID',
           'Rows dropped — primary key collision, cannot validate integrity', _dup_id_rows,
           'Source-system ID collision; <1% of dataset — excluded for data integrity')

df.insert(0, 'Row_ID', range(1, len(df) + 1))

print(f"Rows after duplicate check:    {len(df):,}")
print(f"Duplicate TXN ID rows dropped: {_dup_id_rows}")

=== Step 5: Duplicates ===

  [5a] All columns: Dropped  →  0 affected
  [5b] Transaction ID: Rows dropped — primary key collision, cannot validate integrity  →  84 affected  (Source-system ID collision; <1% of dataset — excluded for data integrity)
Rows after duplicate check:    9,507
Duplicate TXN ID rows dropped: 84


## 6. Fix Invalid Entries

In [37]:
print("=== Step 6: Fix Invalid Entries ===\n")

# 6a — Payment Method: ERR_PM_102 is a source-system error code, not a valid value
_before = int((df['Payment Method'] == 'ERR_PM_102').sum())
df['Payment Method'] = df['Payment Method'].replace('ERR_PM_102', 'Unknown')
log_change('6a', 'Payment Method', "Error code 'ERR_PM_102'",
           "Replaced with 'Unknown'", _before,
           'Error code present in raw data — treated as missing payment method')

print("Payment Method counts after fix:")
print(df['Payment Method'].value_counts())

=== Step 6: Fix Invalid Entries ===

  [6a] Payment Method: Replaced with 'Unknown'  →  26 affected  (Error code present in raw data — treated as missing payment method)
Payment Method counts after fix:
Payment Method
Credit Card       3696
Digital Wallet    2855
Cash              2431
Unknown            525
Name: count, dtype: int64


## 7. Standardise Formats

In [38]:
print("=== Step 7: Standardise Formats ===\n")

# ── 7a: Standardise Item names ────────────────────────────────────────────────
ITEM_MAP = {
    # Coffee variants
    'coffee': 'Coffee', 'cofee': 'Coffee', 'coffe': 'Coffee',
    'Cofee':  'Coffee', 'C0ffee': 'Coffee',   # C0ffee has a zero, not an O
    # Tea variants
    'Tee': 'Tea', 'tee': 'Tea', 'TEA': 'Tea',
    # Donut variants
    'donut': 'Donut', 'Donutt': 'Donut', 'Doughnut': 'Donut',
    # Juice variants
    'juice': 'Juice', 'Juic': 'Juice', 'Juicee': 'Juice',
    # Sandwich variants
    'sandwich': 'Sandwich', 'Sandwhich': 'Sandwich',
}
df['Item'] = df['Item'].str.strip()
_dirty = int(df['Item'].isin(ITEM_MAP).sum())
df['Item'] = df['Item'].replace(ITEM_MAP)
log_change('7a', 'Item', 'Typos / case variants',
           'Mapped to canonical name', _dirty,
           f"{len(ITEM_MAP)} dirty variants normalised to 5 canonical products")

print("Item counts after standardisation:")
print(df['Item'].value_counts())

# ── 7b: Standardise Province names ───────────────────────────────────────────
PROVINCE_MAP = {
    # British Columbia
    'british columbia':  'British Columbia', 'BritishColumbia': 'British Columbia',
    'B.C.':              'British Columbia', 'BC':              'British Columbia',
    'BRITISH COLUMBIA':  'British Columbia', 'British Columba': 'British Columbia',
    'British Columbi':   'British Columbia',
    # Newfoundland and Labrador
    'Newfoundland':              'Newfoundland and Labrador',
    'New Foundland':             'Newfoundland and Labrador',
    'NFLD':                      'Newfoundland and Labrador',
    'NL':                        'Newfoundland and Labrador',
    'newfoundland and labrador': 'Newfoundland and Labrador',
    'NEWFOUNDLAND':              'Newfoundland and Labrador',
    'newfoundland':              'Newfoundland and Labrador',
    'Newfoundlan':               'Newfoundland and Labrador',
    # Saskatchewan
    'Saskatchewn':  'Saskatchewan', 'Sask.':        'Saskatchewan',
    'SK':           'Saskatchewan', 'SASKATCHEWAN': 'Saskatchewan',
    'Sasktchewan':  'Saskatchewan', 'Saskatchewa':  'Saskatchewan',
    'saskatchewan': 'Saskatchewan',
    # Manitoba
    'Manitba':  'Manitoba', 'MB':        'Manitoba',
    'MANITOBA': 'Manitoba', 'Manitobaa': 'Manitoba',
    'Manitob':  'Manitoba', 'mb':        'Manitoba',
    'manitoba': 'Manitoba',
    # Ontario — not an expected province; retained and flagged below
    'Ont.':    'Ontario', 'ON':      'Ontario',
    'ontario': 'Ontario', 'Ontaroi': 'Ontario',
    'Ontairo': 'Ontario',
}
df['Province'] = df['Province'].str.strip()
_dirty = int(df['Province'].isin(PROVINCE_MAP).sum())
df['Province'] = df['Province'].replace(PROVINCE_MAP)
log_change('7b', 'Province', 'Abbreviations / typos / case variants',
           'Mapped to full official province name', _dirty,
           'Ontario rows retained but outside expected 4-province scope — flag for team review')

_ontario = int((df['Province'] == 'Ontario').sum())
if _ontario > 0:
    print(f"\nWARNING: {_ontario} row(s) have Province = 'Ontario' — verify if in scope")

print("\nProvince counts after standardisation:")
print(df['Province'].value_counts(dropna=False))

# ── 7c: Parse Transaction Date to datetime64 ──────────────────────────────────
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
_new_nat = int(df['Transaction Date'].isnull().sum())
log_change('7c', 'Transaction Date', 'Stored as string/object dtype',
           'Parsed to datetime64', len(df),
           f'{_new_nat} row(s) became NaT from unparseable strings')
if _new_nat > 0:
    _before = len(df)
    df.dropna(subset=['Transaction Date'], inplace=True)
    log_change('7c-drop', 'Transaction Date', 'Unparseable date string → NaT',
               'Row dropped', _before - len(df))

print(f"\nDate range: {df['Transaction Date'].min().date()} → {df['Transaction Date'].max().date()}")

# ── 7d: Cast Quantity to int64 ────────────────────────────────────────────────
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
_new_nan = int(df['Quantity'].isnull().sum())
if _new_nan > 0:
    _before = len(df)
    df.dropna(subset=['Quantity'], inplace=True)
    log_change('7d', 'Quantity', 'Non-numeric value coerced to NaN',
               'Row dropped', _before - len(df))
df['Quantity'] = df['Quantity'].astype(int)
log_change('7e', 'Quantity', 'Object dtype', 'Cast to int64', len(df),
           'Entire column converted')

print(f"\nQuantity dtype: {df['Quantity'].dtype}  |  range: {df['Quantity'].min()}-{df['Quantity'].max()}")
print(f"Rows remaining: {len(df):,}")

=== Step 7: Standardise Formats ===

  [7a] Item: Mapped to canonical name  →  1,226 affected  (16 dirty variants normalised to 5 canonical products)
Item counts after standardisation:
Item
Coffee       2666
Tea          1505
Sandwich     1228
Donut        1072
Cookie        913
Refresher     771
Salad         721
Juice         631
Name: count, dtype: int64
  [7b] Province: Mapped to full official province name  →  3,360 affected  (Ontario rows retained but outside expected 4-province scope — flag for team review)


Province counts after standardisation:
Province
British Columbia             3122
Newfoundland and Labrador    2266
Saskatchewan                 1882
Manitoba                     1823
Unknown                       396
Ontario                        18
Name: count, dtype: int64
  [7c] Transaction Date: Parsed to datetime64  →  9,507 affected  (24 row(s) became NaT from unparseable strings)
  [7c-drop] Transaction Date: Row dropped  →  24 affected

Date range: 2023-01-01 → 20

## 8. Validate Derived Columns & Check for Outliers

In [39]:
print("=== Step 8: Validate Derived Columns & Check for Outliers ===\n")

# 8a — Total Spent must equal Quantity × Price Per Unit
df['_calc_total'] = df['Quantity'] * df['Price Per Unit']
_mismatches = int((abs(df['_calc_total'] - df['Total Spent']) > 0.01).sum())
log_change('8a', 'Total Spent',
           f'{_mismatches} mismatch(es) vs Quantity × Price Per Unit',
           'No action required' if _mismatches == 0 else 'FLAGGED — review mismatched rows',
           _mismatches,
           'Derived column validated post-cleaning')
df.drop(columns=['_calc_total'], inplace=True)

# 8b — All dates should fall within 2023
_out_of_year = int((df['Transaction Date'].dt.year != 2023).sum())
log_change('8b', 'Transaction Date',
           f'{_out_of_year} date(s) outside 2023',
           'No action required' if _out_of_year == 0 else 'FLAGGED — dates outside fiscal year',
           _out_of_year,
           'Dataset is expected to cover fiscal year 2023 only')

# 8c — Quantity should be in expected café range (1–5)
_out_of_range = int((~df['Quantity'].between(1, 5)).sum())
log_change('8c', 'Quantity',
           f'{_out_of_range} value(s) outside expected range 1-5',
           'No action required' if _out_of_range == 0 else 'FLAGGED — unexpected quantities',
           _out_of_range,
           'Café orders expected to be 1-5 items per transaction')

print("All validation checks complete.")

=== Step 8: Validate Derived Columns & Check for Outliers ===

  [8a] Total Spent: No action required  →  0 affected  (Derived column validated post-cleaning)
  [8b] Transaction Date: No action required  →  0 affected  (Dataset is expected to cover fiscal year 2023 only)
  [8c] Quantity: No action required  →  0 affected  (Café orders expected to be 1-5 items per transaction)
All validation checks complete.


## 9. Save Clean Dataset & Export Audit Log

In [40]:
OUTPUT_PATH = '../data/clean/data_clean.csv'
df.to_csv(OUTPUT_PATH, index=False)

rows_removed = _original_rows - len(df)
print(f"Saved: {OUTPUT_PATH}")
print(f"Final shape:   {len(df):,} rows × {df.shape[1]} columns")
print(f"Rows removed:  {rows_removed:,}  ({rows_removed / _original_rows * 100:.1f}% of original)\n")

print("=" * 65)
print("FULL AUDIT LOG")
print("=" * 65)
log_df = pd.DataFrame(cleaning_log)
display(log_df)

Saved: ../data/clean/data_clean.csv
Final shape:   9,483 rows × 10 columns
Rows removed:  517  (5.2% of original)

FULL AUDIT LOG


,Step,Column,Issue,Action,Rows Affected,Notes
0,4a,All columns,Empty / whitespace-only string cells,Replaced with NaN,397,Regex replacement catches both empty strings a...
1,4b,Item,Null item name,Row dropped,100,Cannot identify product — row has no analytica...
2,4c,Quantity,Null quantity,Row dropped,98,Revenue calculation requires quantity
3,4d,Transaction Date,Null date,Row dropped,211,Date is required for forecasting — row excluded
4,4e,Payment Method,Null payment method,Filled with 'Unknown',507,Payment channel not always recorded; row retained
5,4f,Location,Null location,Filled with 'Unknown',844,Location not always captured; row retained
6,4g,Province,Null province,Filled with 'Unknown',396,Will be standardised in step 7
7,5a,All columns,Fully duplicate rows,Dropped,0,
8,5b,Transaction ID,84 rows share a non-unique Transaction ID,"Rows dropped — primary key collision, cannot v...",84,Source-system ID collision; <1% of dataset — e...
9,6a,Payment Method,Error code 'ERR_PM_102',Replaced with 'Unknown',26,Error code present in raw data — treated as mi...


In [41]:
# Export audit log so it can be referenced in docs and Power BI
log_df = pd.DataFrame(cleaning_log)
log_df.to_csv('../data/clean/cleaning_audit_log.csv', index=False)
print("Audit log saved: ../data/clean/cleaning_audit_log.csv\n")

print(f"Total cleaning actions recorded: {len(cleaning_log)}")
print(f"\nRows affected by action type:")
print(log_df.groupby('Action')['Rows Affected'].sum().sort_values(ascending=False).to_string())

Audit log saved: ../data/clean/cleaning_audit_log.csv

Total cleaning actions recorded: 18

Rows affected by action type:
Action
Parsed to datetime64                                               9507
Cast to int64                                                      9483
Mapped to full official province name                              3360
Filled with 'Unknown'                                              1747
Mapped to canonical name                                           1226
Row dropped                                                         433
Replaced with NaN                                                   397
Rows dropped — primary key collision, cannot validate integrity      84
Replaced with 'Unknown'                                              26
Dropped                                                               0
No action required                                                    0


In [42]:
df.describe(include='all')

,Row_ID,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Province
count,9483.000000,9483,9483,9483.000000,9483.000000,9483.000000,9483,9483,9483,9483
unique,NaN,9483,8,NaN,NaN,NaN,4,3,NaN,6
top,NaN,TXN_9687814,Coffee,NaN,NaN,NaN,Credit Card,In-store,NaN,British Columbia
freq,NaN,1,2660,NaN,NaN,NaN,3686,5611,NaN,3114
mean,4755.287673,NaN,NaN,1.988506,4.342929,8.639935,NaN,NaN,2023-07-01 20:51:05.865232,NaN
min,1.000000,NaN,NaN,1.000000,2.000000,2.000000,NaN,NaN,2023-01-01 00:00:00,NaN
25%,2380.500000,NaN,NaN,1.000000,3.000000,3.500000,NaN,NaN,2023-03-31 00:00:00,NaN
50%,4755.000000,NaN,NaN,2.000000,3.500000,7.000000,NaN,NaN,2023-06-30 00:00:00,NaN
75%,7132.500000,NaN,NaN,3.000000,5.000000,10.500000,NaN,NaN,2023-10-02 00:00:00,NaN
max,9507.000000,NaN,NaN,5.000000,9.000000,45.000000,NaN,NaN,2023-12-31 00:00:00,NaN
